# Module 5: RAG
# Topic 46: Retrieval-Augmented Generation (RAG) using LangChain

> **Interview Difficulty:** ⭐⭐⭐⭐⭐ (Most Important Topic)
>
> **Interview Frequency:** Extremely High
>
> **Prerequisites:**
> - Document Loaders ✅
> - Text Splitters ✅
> - Embeddings ✅
> - Vector Stores ✅
> - Retrievers ✅

---

# Learning Objectives

After this topic, you should be able to answer:

- What is RAG?
- Why is RAG needed?
- Complete RAG Architecture
- Indexing Pipeline
- Retrieval Pipeline
- LangChain RAG Implementation
- End-to-End Code Example
- Common Challenges
- Best Practices
- Interview Questions

---

# 1. What is RAG?

**RAG (Retrieval-Augmented Generation)** is a technique where an LLM retrieves relevant information from an external knowledge source before generating a response.

Instead of relying only on its training data, the LLM uses **retrieved documents** as additional context.

---

# Interview Definition ⭐⭐⭐⭐⭐

> **Retrieval-Augmented Generation (RAG) is an architecture that combines information retrieval with Large Language Models by fetching relevant documents from an external knowledge base and using them as context to generate accurate, up-to-date, and grounded responses.**

---

# 2. Why Do We Need RAG?

Suppose GPT was trained until 2024.

User asks:

```
What is our company's leave policy?
```

The LLM doesn't know because the information is private.

Without RAG

```
LLM

↓

"I don't know"
```

With RAG

```
LLM

+

Company Documents

↓

Correct Answer
```

---

# 3. Problems Solved by RAG

Without RAG

- Hallucinations
- No private knowledge
- Outdated information
- No citations
- No enterprise data access

With RAG

- Access private documents
- Up-to-date information
- Grounded answers
- Source citations
- Reduced hallucinations

---

# 4. Complete RAG Architecture

```text
                INDEXING

PDF / DOCX / Website

↓

Document Loader

↓

Text Splitter

↓

Chunks

↓

Embedding Model

↓

Vectors

↓

Vector Database

===================================

               RETRIEVAL

User Question

↓

Embedding Model

↓

Retriever

↓

Top-K Chunks

↓

Prompt

↓

LLM

↓

Final Answer
```

---

# 5. Two Phases of RAG

## Phase 1: Indexing

Performed once (or when documents change).

```text
Documents

↓

Load

↓

Split

↓

Embed

↓

Store
```

---

## Phase 2: Retrieval

Performed for every user query.

```text
Question

↓

Embed

↓

Retrieve

↓

Generate Answer
```

---

# 6. Indexing Pipeline

```python
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

loader = PyPDFLoader("employee_handbook.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
```

At this stage:

```
PDF

↓

Chunks

↓

Embeddings
```

---

# 7. Store Chunks in Vector Database

Example with Qdrant:

```python
from langchain_qdrant import QdrantVectorStore

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url="http://localhost:6333",
    collection_name="hr_policy"
)
```

Now the vector database contains

- Chunks
- Embeddings
- Metadata

---

# 8. Retrieval Pipeline

```python
retriever = vector_store.as_retriever(
    search_kwargs={
        "k":3
    }
)
```

Retrieve relevant chunks

```python
docs = retriever.invoke(
    "What is the leave policy?"
)
```

---

# 9. Prompt Construction

The retrieved chunks are inserted into the prompt.

Example

```text
You are an HR assistant.

Answer ONLY using the provided context.

Context

--------------------

Chunk 1

Chunk 2

Chunk 3

--------------------

Question

What is the leave policy?
```

The LLM now answers using retrieved context instead of guessing.

---

# 10. LangChain RAG Chain

Using LCEL:

```python
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    Answer the question using only the context.

    Context:
    {context}

    Question:
    {question}
    """
)

chain = (
    {
        "context": retriever,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)
```

Invoke

```python
response = chain.invoke(
    "What is the leave policy?"
)

print(response)
```

---

# 11. End-to-End Workflow

```text
Employee Handbook.pdf

↓

Loader

↓

Splitter

↓

Chunks

↓

Embeddings

↓

Qdrant

==================

User Question

↓

Embedding

↓

Retriever

↓

Top 3 Chunks

↓

Prompt

↓

LLM

↓

Answer
```

---

# 12. RAG vs Fine-Tuning

A very common interview question.

| RAG | Fine-Tuning |
|------|-------------|
| External knowledge | Internal model weights |
| Easy to update | Requires retraining |
| Good for changing data | Good for behavior/style |
| Fast document updates | Expensive updates |
| Documents remain separate | Knowledge embedded into model |

---

# 13. RAG vs Memory

| RAG | Memory |
|------|--------|
| Retrieves documents | Stores conversations |
| External knowledge | User interactions |
| Enterprise data | Chat history |
| Semantic search | Conversation continuity |

---

# 14. RAG vs Search Engine

| Search Engine | RAG |
|---------------|-----|
| Returns documents | Returns answers |
| User reads results | LLM synthesizes response |
| Keyword/semantic search | Retrieval + generation |

---

# 15. Enterprise Example

Payroll Assistant

User

```
How many casual leaves do I have?
```

Flow

```text
Question

↓

Retriever

↓

Employee Policy

↓

Leave Rules

↓

LLM

↓

Answer
```

Instead of hallucinating, the assistant uses the policy documents.

---

# 16. Common Challenges

### Hallucination

Cause

```
Poor retrieved context
```

Solution

```
Improve retrieval quality
```

---

### Irrelevant Chunks

Cause

```
Poor chunking
```

Solution

```
Tune chunk size and overlap
```

---

### Missing Context

Cause

```
Top-K too small
```

Solution

```
Increase K or improve embeddings
```

---

### Duplicate Chunks

Cause

```
Similarity search
```

Solution

```
Use MMR
```

---

# 17. Production Best Practices

✅ Use metadata filtering.

✅ Choose chunk sizes carefully.

✅ Use the same embedding model for indexing and querying.

✅ Store source metadata for citations.

✅ Evaluate retrieval quality separately from LLM quality.

✅ Cache embeddings for static documents.

---

# 18. Common Mistakes

❌ Embedding entire PDFs.

❌ Ignoring metadata.

❌ Using different embedding models.

❌ Setting Top-K too high.

❌ Blaming the LLM when retrieval is poor.

---

# 19. Interview Questions

## Q1. What is RAG?

**Answer:**

RAG combines document retrieval with LLM generation by retrieving relevant information from an external knowledge base before generating an answer.

---

## Q2. Why is RAG preferred over Fine-Tuning for enterprise knowledge?

**Answer:**

Enterprise documents change frequently. With RAG, documents can be updated in the vector database without retraining the model, making updates faster, cheaper, and more maintainable.

---

## Q3. What are the two phases of RAG?

**Answer:**

1. **Indexing** – Load, split, embed, and store documents.
2. **Retrieval** – Embed the query, retrieve relevant chunks, and generate the answer.

---

## Q4. Why is chunking important?

**Answer:**

Chunking improves retrieval quality by breaking large documents into meaningful pieces that fit within the model's context window.

---

## Q5. What happens if retrieval is poor?

**Answer:**

The LLM receives irrelevant or incomplete context, leading to inaccurate or hallucinated responses, even if the LLM itself is highly capable.

---

## Q6. How can you improve RAG performance?

**Answer:**

Improve document quality, optimize chunk size and overlap, use a strong embedding model, tune Top-K, apply metadata filtering, use MMR, and evaluate retrieval quality with metrics like Recall@K.

---

# 20. Real Project Architecture (Interview Ready)

This closely resembles many production enterprise chatbots.

```text
PDF Upload

↓

PyPDFLoader

↓

RecursiveCharacterTextSplitter

↓

OpenAI Embeddings

↓

Qdrant

↓

Retriever

↓

Prompt Template

↓

GPT-4 / GPT-4.1 / GPT-5

↓

Answer with Source
```

---

# 21. Quick Revision

| Component | Responsibility |
|------------|----------------|
| Loader | Read documents |
| Splitter | Create chunks |
| Embeddings | Convert text to vectors |
| Vector DB | Store vectors |
| Retriever | Fetch relevant chunks |
| Prompt | Combine context + question |
| LLM | Generate final answer |

---

# Interview Cheat Sheet

```text
INDEXING

Documents

↓

Loader

↓

Splitter

↓

Embeddings

↓

Vector DB

=====================

RETRIEVAL

Question

↓

Embedding

↓

Retriever

↓

Top-K Chunks

↓

Prompt

↓

LLM

↓

Answer
```

---

# 22. Real Interview Scenario

**Question:**

> Your RAG chatbot sometimes gives incorrect answers even though the correct information exists in the documents. How would you debug it?

**Answer:**

I would debug the pipeline step by step:

1. Verify that the document was loaded correctly.
2. Check whether chunking split important information incorrectly.
3. Confirm the embedding model used for indexing matches the one used for querying.
4. Inspect the retrieved Top-K chunks to see whether the relevant context is returned.
5. Tune retrieval parameters such as `k`, chunk size, overlap, and metadata filters.
6. Only after validating retrieval would I investigate the prompt or the LLM.

This isolates whether the issue is in indexing, retrieval, or generation.

---

# 30-Second Interview Answer

> **RAG is a two-stage architecture consisting of indexing and retrieval. During indexing, documents are loaded, split into chunks, converted into embeddings, and stored in a vector database. During retrieval, the user's query is embedded, the most relevant chunks are fetched using semantic search, and those chunks are included in the prompt sent to the LLM. This enables accurate, grounded responses using private or frequently changing knowledge without retraining the model.**

---

# Key Takeaway

> **In most production RAG systems, retrieval quality has a greater impact on answer quality than the choice of LLM. A well-designed indexing pipeline, effective retrieval strategy, and strong prompt construction are the foundation of a successful RAG application.**